# Image Indexing Walkthrough — Sections 7.1–7.3

Demonstrates how images are extracted, saved to disk, and linked to text chunks in the
vector index.  Uses the handlebar chapter (ch. 7) as a concrete example because it has
clearly labelled diagram callouts and spans two pages with interesting section boundaries.

**Strategy:** images are stored as files under `data/images/` and their paths are written
into ChromaDB metadata alongside the text chunk.  At retrieval time the agent gets both
text and the ordered list of diagram paths — no separate image embedding required.

In [ ]:
import sys, json
sys.path.insert(0, "..")

import chromadb
from IPython.display import display, Image as IPImage, Markdown

DB_PATH = "../data/chroma"

col = chromadb.PersistentClient(path=DB_PATH).get_collection("manuals")

# Fetch all chunks for sections 7.1, 7.2, 7.3 from ChromaDB
raw = col.get(
    where={"$and": [{"chapter_num": 7}, {"section": {"$in": ["7.1", "7.2", "7.3"]}}]},
    include=["metadatas", "documents"],
)

# Sort by section for consistent display
sections = sorted(
    zip(raw["metadatas"], raw["documents"]),
    key=lambda x: x[0]["section"],
)

print(f"Chunks retrieved: {len(sections)}")

## 1. Chunk overview

Each chunk carries `page_nums` (the source pages) and `image_paths` (ordered file paths
for every image on those pages).  Images are listed in **content-stream order** — the same
sequence they appear visually on the page — so diagram callout letters/numbers stay in sync.

In [ ]:
print(f"{'Section':<8} {'Title':<42} {'Pages':<12} {'Images'}")
print("-" * 75)
for meta, _ in sections:
    title = meta["section_title"] or meta.get("chapter_title", "")
    pages = json.loads(meta["page_nums"])
    imgs  = json.loads(meta["image_paths"])
    print(f"{meta['section']:<8} {title[:42]:<42} {str(pages):<12} {len(imgs)}")

## 2. Section 7.1 — Handlebar position

Single page (p. 50).  The text references callout **(A)** — the hole distance measurement
marked on the diagram.  Both images come from page 50 in content-stream order.

In [ ]:
meta_71, text_71 = next((m, d) for m, d in sections if m["section"] == "7.1")
imgs_71 = json.loads(meta_71["image_paths"])

print(f"Pages   : {json.loads(meta_71['page_nums'])}")
print(f"Images  : {len(imgs_71)}")
print(f"\nText excerpt:")
print(text_71[:300])

In [ ]:
for i, path in enumerate(imgs_71):
    display(Markdown(f"**Image {i+1} of {len(imgs_71)}** — `{path.split('/')[-1]}`"))
    display(IPImage(filename=f"../{path}", width=400))

## 3. Section 7.2 — Adjusting the handlebar position

Spans two pages (pp. 50–51).  The chunk picks up images from **both** pages:
the two diagrams from page 50 (shared with 7.1) plus a third from page 51.
The text callout **(1)** refers to the handlebar clamp screws visible in the first diagram.

In [ ]:
meta_72, text_72 = next((m, d) for m, d in sections if m["section"] == "7.2")
imgs_72 = json.loads(meta_72["image_paths"])

print(f"Pages   : {json.loads(meta_72['page_nums'])}")
print(f"Images  : {len(imgs_72)}")
print(f"\nText excerpt:")
print(text_72[:400])

In [ ]:
for i, path in enumerate(imgs_72):
    display(Markdown(f"**Image {i+1} of {len(imgs_72)}** — `{path.split('/')[-1]}`"))
    display(IPImage(filename=f"../{path}", width=400))

## 4. Section 7.3 — Adjusting the basic position of the clutch lever

Starts on page 51.  One diagram showing adjusting screw **(1)**.
This image is also linked to section 7.2 above because both sections share page 51.

In [ ]:
meta_73, text_73 = next((m, d) for m, d in sections if m["section"] == "7.3")
imgs_73 = json.loads(meta_73["image_paths"])

print(f"Pages   : {json.loads(meta_73['page_nums'])}")
print(f"Images  : {len(imgs_73)}")
print(f"\nText excerpt:")
print(text_73[:300])

In [ ]:
for i, path in enumerate(imgs_73):
    display(Markdown(f"**Image {i+1} of {len(imgs_73)}** — `{path.split('/')[-1]}`"))
    display(IPImage(filename=f"../{path}", width=400))

## 5. Image sharing across section boundaries

When a page contains content from more than one section, every section that includes that
page in its `page_nums` receives the full set of images from it.  The table below shows
which images are shared between sections.

This is a deliberate trade-off: it is better to surface an image in both adjacent sections
than to risk omitting it from the section that actually references it.

In [ ]:
from collections import defaultdict

# Build a map: filename → which sections reference it
file_to_sections: dict[str, list[str]] = defaultdict(list)
for meta, _ in sections:
    for path in json.loads(meta["image_paths"]):
        file_to_sections[path.split("/")[-1]].append(meta["section"])

print(f"{'File':<20}  {'Referenced by'}")
print("-" * 45)
for fname, secs in sorted(file_to_sections.items()):
    shared = " ← shared" if len(secs) > 1 else ""
    print(f"{fname:<20}  {', '.join(secs)}{shared}")